In [ ]:
from jaxtyping import Float, Array, Key, Scalar
import jax
import jax.numpy as jnp
from jax.numpy.fft import fftfreq, ifft, fft
import jax.scipy as jsp
import jax.random as jr
from flax import nnx
import optax
from einops import rearrange, einsum

import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from corner import corner
from emcee import EnsembleSampler

In [ ]:
Parameters = Float[Array, "sources 3"]
Observation = Float[Array, "times 2"]

rngs = nnx.Rngs(42)


SOURCES = 2
CHANNELS = 2
AMPLITUDE_RANGE = (1, 10)
FREQUENCY_RANGE = (1, 10)
PHASE_RANGE = (0, 2 * np.pi)

TOBS=10
NTIMESAMPS=1024

TIMES = jnp.linspace(0, TOBS, NTIMESAMPS)[..., None]

def noise_PSD(f,A_psd=1.,f_ref=3,alpha=0):
    return A_psd * jnp.where(f != 0, (f / f_ref) ** alpha, 0.0)

def noise_gen(rng: Key):
    T=TOBS
    N=NTIMESAMPS
    dt=T/N

    # Frequency bins
    freqs = fftfreq(N, dt)
    S = jnp.zeros(N)

    S  = S.at[1:].set(noise_PSD(jnp.abs(freqs[1:])))   # indices 1 … N-1
    S  = S.at[0].set(0.0)
    if N % 2 == 0:                               # Nyquist bin if N even
        S  = S.at[N // 2].set(0.0)

    rng_r, rng_i = jr.split(rng, 2)
    xf = jnp.sqrt(S*N)[:, None] * (jr.normal(rng_r, shape=(N,CHANNELS))
                                   + 1j * jr.normal(rng_i, shape=(N,CHANNELS)) )/jnp.sqrt(2.0)

    # Impose Hermitian symmetry for k<0
    idx = jnp.arange(1, N // 2)
    xf  = xf.at[N - idx].set(jnp.conj(xf[idx]))
    
    return jnp.real(ifft(xf,axis=0))



@jax.jit
def sample_joint(rng: Key) -> tuple[Parameters, Observation]:
    rng_a, rng_f, rng_phi, rng_y = jr.split(rng, 4)
    log_a = jr.uniform(
        rng_a,
        shape=(SOURCES,),
        minval=np.log(AMPLITUDE_RANGE[0]),
        maxval=np.log(AMPLITUDE_RANGE[1]),
    )
    log_f = jr.uniform(
        rng_f,
        shape=(SOURCES,),
        minval=np.log(FREQUENCY_RANGE[0]),
        maxval=np.log(FREQUENCY_RANGE[1]),
    )
    phi = jr.uniform(
        rng_phi,
        shape=(SOURCES,),
        minval=PHASE_RANGE[0],
        maxval=PHASE_RANGE[1],
    )
    a = jnp.exp(log_a)
    f = jnp.exp(log_f)
    x = jnp.stack([a, f, phi], axis=-1)

    theta = 2 * jnp.pi * f * TIMES + phi
    h_plus = (a * jnp.sin(theta)).sum(-1)
    h_cross = (a * jnp.cos(theta)).sum(-1)
    h = jnp.stack([h_plus, h_cross], axis=-1)

    y = h + noise_gen(rng_y)
    return x, y


@jax.jit
def log_posterior(x_flat: Parameters, y: Observation) -> Scalar:
    x = rearrange(x_flat, "... (sources p) -> ... sources p", p=3)
    a, f, phi = x[..., 0], x[..., 1], x[..., 2]
    log_a, log_f = jnp.log(a), jnp.log(f)

    theta = 2 * jnp.pi * f * TIMES + phi
    h_plus = (a * jnp.sin(theta)).sum(-1)
    h_cross = (a * jnp.cos(theta)).sum(-1)
    h = jnp.stack([h_plus, h_cross], axis=-1)

    noisevars = noise_PSD(jnp.abs(fftfreq(NTIMESAMPS,TOBS/NTIMESAMPS)))[1:,None]*NTIMESAMPS
    residual = jnp.abs(fft(y - h,axis=0)[1:])
    log_likelihood = -einsum(residual**2 / (2*noisevars), "... t c-> ...")

    mask_a = (AMPLITUDE_RANGE[0] < a) * (a < AMPLITUDE_RANGE[1])
    mask_f = (FREQUENCY_RANGE[0] < f) * (f < FREQUENCY_RANGE[1])
    mask_phi = (PHASE_RANGE[0] < phi) * (phi < PHASE_RANGE[1])
    log_prior = jnp.where(mask_a * mask_f * mask_phi, -log_a - log_f, -jnp.inf).sum(-1)
    return log_prior + log_likelihood

In [ ]:
NUM_BLOCKS = 4
NUM_HEADS = 8
HIDDEN_DIM = 64 * NUM_HEADS
PATCH_SIZE = 32
LEARNING_RATE = 1e-4
BATCH_SIZE = 512
TOTAL_EXAMPLES = 1024_0000


def adaptive_norm(
    x: Float[Array, "... N D"],
    scale: Float[Array, "... 1 D"],
    shift: Float[Array, "... 1 D"],
):
    x = x - x.mean(axis=-1, keepdims=True)
    x = x / x.std(axis=-1, keepdims=True)
    x = x * (1 + scale) + shift
    return x


class Modulation(nnx.Module):
    def __init__(self, dim: int, *, rngs: nnx.Rngs):
        self.linear = nnx.LinearGeneral(
            in_features=dim,
            out_features=(1, 3 * dim),
            kernel_init=nnx.initializers.zeros,
            bias_init=nnx.initializers.zeros,
            rngs=rngs,
        )

    def __call__(self, y: Float[Array, "... D"]) -> tuple[Float[Array, "... 1 D"], ...]:
        modulation = self.linear(nnx.silu(y))
        shift, scale, gate = jnp.split(modulation, 3, axis=-1)
        return shift, scale, gate


class SinusoidalEmbed(nnx.Module):
    def __init__(self, dim: int, period: float = 2 * np.pi, *, rngs: nnx.Rngs):
        self.dim = dim
        self.period = period
        self.embed = FeedForward(2 * dim, dim, dim, rngs=rngs)

    def __call__(self, t: Float[Array, "..."]) -> Float[Array, "... D"]:
        freqs = jnp.exp(-jnp.log(self.period) * jnp.linspace(0, 1, self.dim))
        angles = 2 * jnp.pi * freqs * t[..., None]
        x = jnp.concat([jnp.sin(angles), jnp.cos(angles)], axis=-1)
        x = self.embed(x)
        return x


class FeedForward(nnx.Sequential):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        activation=nnx.silu,
        *,
        rngs: nnx.Rngs,
    ):
        super().__init__(
            nnx.Linear(input_dim, hidden_dim, rngs=rngs),
            activation,
            nnx.Linear(hidden_dim, output_dim, rngs=rngs),
        )


class CrossAttention(nnx.Module):
    def __init__(self, dim: int, num_heads: int, use_bias=False, *, rngs: nnx.Rngs):
        super().__init__()
        assert dim % num_heads == 0, "dim should be divisible by num_heads"
        self.num_heads = num_heads
        self.qkv_proj_x = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
        self.qkv_proj_c = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
        self.out_proj_x = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)
        self.out_proj_c = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)

    def __call__(self, x: Float[Array, "... N D"], c: Float[Array, "... M D"]):
        *_, N, Dx = x.shape
        *_, M, Dc = c.shape
        assert Dx == Dc, "x and c should have the same feature dimension"

        x = self.qkv_proj_x(x)
        c = self.qkv_proj_c(c)
        h = jnp.concat([x, c], axis=-2)
        qkv = rearrange(h, "... N (H D) -> ... N H D", H=self.num_heads)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        h = nnx.dot_product_attention(q, k, v)
        h = rearrange(h, "... N H D -> ... N (H D)")
        x, c = jnp.split(h, [N], axis=-2)
        x = self.out_proj_x(x)
        c = self.out_proj_c(c)
        return x, c


class MMDiTBlock(nnx.Module):
    def __init__(self, dim: int, num_heads: int, expand: int = 4, *, rngs: nnx.Rngs):
        self.modulation1_x = Modulation(dim, rngs=rngs)
        self.modulation1_c = Modulation(dim, rngs=rngs)
        self.modulation2_x = Modulation(dim, rngs=rngs)
        self.modulation2_c = Modulation(dim, rngs=rngs)
        self.attention = CrossAttention(dim, num_heads, rngs=rngs)
        self.mlp_x = FeedForward(dim, expand * dim, dim, rngs=rngs)
        self.mlp_c = FeedForward(dim, expand * dim, dim, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "... N D"],
        c: Float[Array, "... M D"],
        y: Float[Array, "... D"],
    ):
        # cross attention block
        shift_x, scale_x, gate_x = self.modulation1_x(y)
        shift_c, scale_c, gate_c = self.modulation1_c(y)
        hx = adaptive_norm(x, scale_x, shift_x)
        hc = adaptive_norm(c, scale_c, shift_c)
        hx, hc = self.attention(hx, hc)
        x = x + hx * gate_x
        c = c + hc * gate_c

        # feed forward blocks
        shift_x, scale_x, gate_x = self.modulation2_x(y)
        hx = adaptive_norm(x, scale_x, shift_x)
        hx = self.mlp_x(hx)
        x = x + hx * gate_x

        shift_c, scale_c, gate_c = self.modulation2_c(y)
        hc = adaptive_norm(c, scale_c, shift_c)
        hc = self.mlp_c(hc)
        c = c + hc * gate_c
        return x, c


class MMDiT(nnx.Module):
    def __init__(
        self,
        x_dim: int,
        c_dim: int,
        hidden_dim: int,
        num_heads: int,
        num_blocks: int,
        *,
        rngs: nnx.Rngs,
    ):
        self.x_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.x_embed = FeedForward(x_dim, hidden_dim, hidden_dim, rngs=rngs)

        self.c_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.c_embed = FeedForward(c_dim, hidden_dim, hidden_dim, rngs=rngs)

        self.y_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.y_embed = FeedForward(hidden_dim, hidden_dim, hidden_dim, rngs=rngs)

        self.blocks = [
            MMDiTBlock(hidden_dim, num_heads, rngs=rngs) for _ in range(num_blocks)
        ]

        self.out_modulation = Modulation(hidden_dim, rngs=rngs)
        self.out_unembed = FeedForward(hidden_dim, hidden_dim, x_dim, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "... N D"],
        c: Float[Array, "... M D"],
        t: Float[Array, "..."],
    ) -> Float[Array, "... N D"]:
        # embeddings
        x_pos = jnp.linspace(0, 1, x.shape[-2])
        x = self.x_embed(x) + self.x_pos_embed(x_pos)
        c_pos = jnp.linspace(0, 1, c.shape[-2])
        c = self.c_embed(c) + self.c_pos_embed(c_pos)
        y = self.y_embed(self.y_pos_embed(t))

        # cross attention blocks
        for block in self.blocks:
            x, c = block(x, c, y)

        # unembedging
        shift, scale, gate = self.out_modulation(y)
        x = adaptive_norm(x, scale, shift)
        x = self.out_unembed(x)
        return x


class Flow(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.DiT = MMDiT(
            x_dim=3,
            c_dim=2 * PATCH_SIZE,
            hidden_dim=HIDDEN_DIM,
            num_heads=NUM_HEADS,
            num_blocks=NUM_BLOCKS,
            rngs=rngs,
        )

    @nnx.jit
    def __call__(self, x: Parameters, t: Scalar, y: Observation) -> Parameters:
        # c = rearrange(y, "... (T P) C -> ... T (P C)", P=PATCH_SIZE)
        _, _, c = jsp.signal.stft(y, nperseg=PATCH_SIZE - 1, axis=-2)
        c = rearrange(c, "... F C T -> ... T (C F)")
        c = jnp.concatenate([c.real, c.imag], axis=-1)
        return self.DiT(x, c, t)

    @nnx.jit
    def ode_step(
        self, x: Parameters, t: Scalar, y: Observation, dt: float
    ) -> Parameters:
        k1 = self(x, t, y)
        k2 = self(x + k1 * dt / 2, t + dt / 2, y)
        k3 = self(x + k2 * dt / 2, t + dt / 2, y)
        k4 = self(x + k3 * dt, t + dt, y)
        x = x + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6
        return x


@jax.jit
def get_train_batch(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
    def phi(t: Scalar, x1: Parameters, x0: Parameters) -> Parameters:
        return x1 * t + x0 * (1 - t)

    def train_sample(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
        rng_xy, rng_x0, rng_t = jr.split(rng, 3)
        x1, y = sample_joint(rng_xy)
        x0 = jr.normal(rng_x0, x1.shape)
        t = jr.uniform(rng_t, minval=0.0, maxval=1.0)

        xt = x1 * t + x0 * (1 - t)
        dx = jax.jacobian(phi)(t, x1, x0)
        return xt, t, y, dx

    return jax.vmap(train_sample)(jr.split(rng, BATCH_SIZE))


@nnx.jit
def train_step(
    model: Flow,
    optimizer: nnx.Optimizer,
    batch: tuple[Parameters, Scalar, Observation, Parameters],
) -> Scalar:
    def loss_fn(model):
        xt, t, y, dx = batch
        return jnp.mean((model(xt, t, y) - dx) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss


flow = Flow(rngs=rngs)
optimizer = nnx.Optimizer(flow, optax.adamw(learning_rate=LEARNING_RATE))

for step in (pbar := tqdm(range(TOTAL_EXAMPLES // BATCH_SIZE))):
    batch = get_train_batch(rngs.batch())
    loss = train_step(flow, optimizer, batch)
    pbar.set_postfix(loss=loss.item())

In [ ]:
RUNS = 10
SAMPLES = 1024
DIFFUSIONSTEPS = 16
MCMCWALKERS = 32
MCMCDISCARD = 1000
MCMCTHIN = 10

def sample_from_flow(y: Observation):
    t = jnp.zeros((SAMPLES,))
    x = jr.normal(rngs.eval(), (SAMPLES, SOURCES, 3))
    y = jnp.broadcast_to(y, (SAMPLES, *y.shape))
    dt = 1.0 / DIFFUSIONSTEPS
    for _ in tqdm(range(DIFFUSIONSTEPS)):
        x = flow.ode_step(x, t, y, dt)
        t += dt
    x = rearrange(x, "... S P -> ... (S P)")
    return np.array(x)


def sample_from_mcmc(y: Observation, x_true_flat: Parameters):
    p0 = x_true_flat*(1+0.001*np.random.randn(MCMCWALKERS//2, x_true_flat.shape[-1]))
    p1 = mirror(x_true_flat)*(1+0.001*np.random.randn(MCMCWALKERS//2, x_true_flat.shape[-1]))
    p0 = np.concatenate([p0,p1],axis=0)
    sampler = EnsembleSampler(MCMCWALKERS, x_true_flat.shape[-1], log_posterior, args=(y,))
    sampler.run_mcmc(p0, nsteps=MCMCTHIN * SAMPLES // MCMCWALKERS + MCMCDISCARD, progress=True)
    x = sampler.get_chain(flat=True, discard=MCMCDISCARD, thin=MCMCTHIN)
    return x


def mirror(x_flat):
    x = rearrange(x_flat, "... (S P) -> ... S P", P=3)
    x_mirrored = x[..., ::-1, :]
    x_mirrored_flat = rearrange(x_mirrored, "... S P -> ... (S P)")
    return x_mirrored_flat


for run in range(RUNS):
    x_true, y = sample_joint(rngs.eval())
    x_true_flat = rearrange(x_true, "... N P -> ... (N P)")

    print("Running flow sampling...")
    #generated_samples = sample_from_flow(y)
    print("Running MCMC...")
    mcmc_samples = sample_from_mcmc(y, x_true_flat)
    print()

    param_names = sum(([f"A_{i}", f"f_{i}", f"$phi_{i}$"]  for i in range(SOURCES)), [])
    fig = corner(mcmc_samples, labels=param_names, truths=x_true_flat, color="blue")
    fig = corner(mirror(mcmc_samples), color="cyan", fig=fig)
    #fig = corner(generated_samples, color="red", fig=fig)
    #fig = corner(mirror(generated_samples), color="pink", fig=fig)
    
    plt.show()